<a href="https://colab.research.google.com/github/Haniiko1/CTL_Assay/blob/main/project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2

In [ ]:
#@title Run this cell to import all neccessary libraries
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import urllib.request
import os
import re
import matplotlib.cm as cm
from google.colab import files
from IPython.display import HTML, display # Import HTML for rich display
import math # Required for calculating subplot grid

# Install xlsxwriter if not already installed
try:
    import xlsxwriter
except ImportError:
    %pip install xlsxwriter
    import xlsxwriter

### 1. Load Data

To begin, please specify the location and format of your protease assay data. For example, is it a CSV file, an Excel spreadsheet, or accessible via a database?

# Instruction on how to upload your files



1.   In your raw data file, rename directly onto the columns with your samples' name

2.   In your raw data file, copy **ONLY your data table** into a new blank Excel file. Make sure that the end time row is not included at the bottom of new blank Excel file. The data table must be all the way at the top and all the way to the left of the Excel file.

3.   **IMPORTANT:** Go to File -> Save As... -> Rename the file to anything you would like -> **File Fortmat** and change it to CSV UTF-8 (Comma delimited) -> Save


### The File Format **MUST** be in exactly CSV UTF-8 (Comma delimited) format; otherwise, it will not work.

4.   Run the code cell below and click on Choose Files, then upload the CSV file you just created.










In [ ]:
#@title  Upload your own CSV file from your computer

print("A file chooser dialog will appear below.")
print("Select your CSV file and wait for the upload to complete.")
print()

# This opens the file upload dialog
uploaded = files.upload()

# The filename is shown after upload completes
uploaded_filename = list(uploaded.keys())[0]

print(f"\n✅ File uploaded: {uploaded_filename}")

# Read the uploaded file into a DataFrame
df = pd.read_csv(uploaded_filename)

# Drop columns where all values are NaN directly after loading
df_cleaned = df.dropna(axis=1, how='all')

print(f"Original DataFrame shape: {df.shape}")
print(f"DataFrame shape after dropping all-NaN columns: {df_cleaned.shape}")

# Columns to remove further, including cycle number and temperature
columns_to_remove_further = ['Cycle Nr.', 'Temp. [°C]']

# Drop the specified columns from the already cleaned DataFrame
df_final = df_cleaned.drop(columns=columns_to_remove_further, errors='ignore')

# Convert 'Time [s]' to 'Time [min]' and make it a whole number
if 'Time [s]' in df_final.columns:
    df_final['Time [min]'] = (df_final['Time [s]'] / 60).astype(int)
    # Optionally, you might want to drop the original 'Time [s]' column
    df_final = df_final.drop(columns=['Time [s]'])

# Reorder columns to make 'Time [min]' the first column
if 'Time [min]' in df_final.columns:
    cols = df_final.columns.tolist()
    cols.remove('Time [min]')
    df_final = df_final[['Time [min]'] + cols]

# Rename columns to handle duplicates by appending _1, _2, etc.
# This assumes pandas has already added .1, .2 for actual duplicates during read_csv
original_columns = df_final.columns.tolist()
renamed_columns = []
counts = {} # To count occurrences of base names
suffix_counter = {} # To generate _1, _2, ... suffixes

# First pass to determine which base names are duplicated
for col in original_columns:
    # Strip any .N or _N suffix pandas might have added or that existed from prior runs
    # re.sub(pattern, replacement, text)
    # Find text matching pattern and replace it with replacement.
    # \.\d+|_\d+ means patterns with dot + numbers OR _ + numbers
    base_name = re.sub(r'(\.\d+|_\d+)$', '', col)
    counts[base_name] = counts.get(base_name, 0) + 1

# Second pass to apply renaming
for col in original_columns:
    base_name = re.sub(r'(\.\d+|_\d+)$', '', col)

    if counts[base_name] > 1:
        # If the base name is duplicated, apply a suffix to ALL occurrences
        suffix_counter[base_name] = suffix_counter.get(base_name, 0) + 1
        renamed_columns.append(f"{base_name}_{suffix_counter[base_name]}")
    else:
        # If the base name is unique, keep its name as is (it might be 'Time [min]' or a unique replicate name)
        renamed_columns.append(col)

df_final.columns = renamed_columns


print(f"Final DataFrame shape after removing '{columns_to_remove_further}' and converting time: {df_final.shape}")
print("Column names found in your file after all initial cleaning and time conversion:")
print(list(df_final.columns))
print()

print(f"Shape (final cleaned): {df_final.shape[0]} rows × {df_final.shape[1]} columns")
print()
print("Column names (final cleaned):")
print(list(df_final.columns))


# Replace NaN values with blank strings for display
display(df_final.fillna(''))



# Only run this code cell below once. If ran twice or more, the minimums of each sample before zeroing will become all 0. If this happens you will have to go back to the previous code cell and upload your file again.

In [ ]:
#@title Find the minimum in each replicates, zero them, and find the overall maximum

print("--- Minimums of each sample (before zeroing) ---")

# Finding the minimum value in each sample
min_values = df_final.drop(columns=['Time [min]'], errors='ignore').min()
print(min_values)

# Get the list of columns to normalize (all except 'Time [min]')
columns_to_normalize = min_values.index.tolist()

# Subtract the minimum values from the corresponding columns in df_final
# We use .loc to ensure alignment by column names
df_final.loc[:, columns_to_normalize] = df_final.loc[:, columns_to_normalize].sub(min_values)

print("\n--- Data after zeroing samples ---")
display(df_final.fillna(''))

print("\n--- New maximums after subtracting baseline ---")
new_max_values = df_final.drop(columns=['Time [min]'], errors='ignore').max()
print(new_max_values)

print("\n--- The largest value among all new maximums ---")
print(new_max_values.max())

In [ ]:
#@title Normalize data to the overall maximum
overall_max = new_max_values.max()
print(f"Overall maximum value used for normalization: {overall_max}")

# Create a copy of df_final to store the normalized data
df_normalized = df_final.copy()

# Normalize all columns except 'Time [min]' by dividing by the overall maximum
columns_to_normalize = df_normalized.columns.drop('Time [min]', errors='ignore')
df_normalized[columns_to_normalize] = df_normalized[columns_to_normalize] / overall_max

print("\n--- Normalized DataFrame ---")
display(df_normalized)

In [ ]:
#@title Calculate Average and Standard Deviation per Replicate Group

# 1. Identify unique replicate groups (base names)
replicate_cols = df_normalized.columns.drop('Time [min]', errors='ignore')

# Extract unique base replicate names while preserving their order of first appearance
seen_base_reps = set()
base_replicates = []
for col in replicate_cols:
    # Strip postfixes from samples
    base_name = re.sub(r'(_\d+)$', '', col)
    if base_name not in seen_base_reps:
        base_replicates.append(base_name)
        seen_base_reps.add(base_name)

# Initialize a dictionary to hold all data, including MultiIndex keys
combined_data = {'Time [min]': df_normalized['Time [min]']}

# 2. Iterate through each base replicate group and calculate combined stats
for base_rep in base_replicates:
    # Find all columns in df_normalized that belong to this base_rep group
    group_cols = [col for col in replicate_cols if col.startswith(base_rep + '_') or col == base_rep]

    # Calculate the row-wise mean and standard deviation for these columns
    means = df_normalized[group_cols].mean(axis=1)
    stds = df_normalized[group_cols].std(axis=1).fillna(0)

    # Add to the combined_data dictionary with tuple keys for MultiIndex
    combined_data[(base_rep, 'Avg')] = means
    combined_data[(base_rep, 'Std Dev')] = stds

# Prepare the column tuples for the MultiIndex (for statistical columns only)
column_tuples_for_multiindex = []
for base_rep in base_replicates:
    column_tuples_for_multiindex.append((base_rep, 'Avg'))
    column_tuples_for_multiindex.append((base_rep, 'Std Dev'))

final_multi_index = pd.MultiIndex.from_tuples(column_tuples_for_multiindex)

# Create a dictionary containing only the MultiIndex-keyed Series for DataFrame construction
stats_data_for_df_construction = {}
for k, v in combined_data.items():
    if isinstance(k, tuple): # Select only the (base_rep, 'Avg'/'Std Dev') tuples
        stats_data_for_df_construction[k] = v

# Create the DataFrame from the statistical data
df_summary_stats_multiindex = pd.DataFrame(stats_data_for_df_construction)

# Ensure the columns are in the correct order and have the MultiIndex assigned
df_summary_stats_multiindex.columns = final_multi_index

# *** Apply x100 scaling to 'Avg' and 'Std Dev' columns and rename them ***
new_columns_for_multiindex = []
for col_tuple in df_summary_stats_multiindex.columns:
    if col_tuple[1] == 'Avg':
        # Scale the 'Avg' values
        df_summary_stats_multiindex[col_tuple] = df_summary_stats_multiindex[col_tuple] * 100
        # Create a new column tuple name for 'Avg'
        new_columns_for_multiindex.append((col_tuple[0], 'Avg (x100)'))
    elif col_tuple[1] == 'Std Dev':
        # Scale the 'Std Dev' values
        df_summary_stats_multiindex[col_tuple] = df_summary_stats_multiindex[col_tuple] * 100
        # Create a new column tuple name for 'Std Dev'
        new_columns_for_multiindex.append((col_tuple[0], 'Std Dev (x100)'))
    else:
        new_columns_for_multiindex.append(col_tuple)

df_summary_stats_multiindex.columns = pd.MultiIndex.from_tuples(new_columns_for_multiindex)

# Insert 'Time [min]' as the very first column with a single-level header
df_summary_stats_multiindex.insert(0, 'Time [min]', df_normalized['Time [min]'].values)

# Create a temporary DataFrame for display to explicitly format 'Time [min]' as string
df_display_temp = df_summary_stats_multiindex.copy()
df_display_temp['Time [min]'] = df_display_temp['Time [min]'].apply(lambda x: f'{x:.0f}') # Format as whole number


print("\n--- Average and Standard Deviation per Replicate Group --- ")
display(df_display_temp)

In [ ]:
#@title Plot all data points

plt.figure(figsize=(12, 7))

# Iterate through each base replicate to plot its 'Avg' against 'Time [min]'
for base_rep in base_replicates:
    # The 'Avg' column is already scaled and renamed in df_summary_stats_multiindex
    avg_col_name = (base_rep, 'Avg (x100)') # Use the new scaled column name
    plt.plot(df_summary_stats_multiindex['Time [min]'], df_summary_stats_multiindex[avg_col_name], label=f'{base_rep} Average (x100)')

plt.xlabel('Time [min]')
plt.ylabel('Average of Sample (x100)') # Keep y-axis label consistent with global scaling
plt.title('Average of Each Sample Group vs. Time (Scaled by 100)')
plt.legend()
plt.grid(True)
plt.show()

# These code cell below can take quite some time depends on how much data you have. Should be under 1 minute most of the time

In [ ]:
#@title Obtain all possible starting points + # of points per slice Slope and $R^2$

# Initialize a list to store regression results
regression_results = []

num_rows = len(df_summary_stats_multiindex)

# Iterate through possible numbers of points per slice, starting from 8 up to num_rows
for points_per_slice in range(8, num_rows + 1):
    # Iterate through every possible starting index for the current points_per_slice
    # An implicit stride of 1 covers "every starting point"
    for start_idx in range(0, num_rows - points_per_slice + 1):
        end_idx = start_idx + points_per_slice

        # Extract the slice of data
        slice_df = df_summary_stats_multiindex.iloc[start_idx:end_idx].copy()

        # Ensure 'Time [min]' is numeric
        x = slice_df['Time [min]'].astype(float)

        # Get the actual start and end time values for the slice
        slice_start_time = x.iloc[0]
        slice_end_time = x.iloc[-1]

        # Iterate through each base replicate group
        # `base_replicates` should be available from previous cells (e.g., E0EHzlMJpAf1)
        for base_rep in base_replicates:
            avg_col_name = (base_rep, 'Avg (x100)') # Use the scaled column name

            # Ensure 'Avg' column is numeric
            y = slice_df[avg_col_name].astype(float)

            # Perform linear regression only if there's enough data (at least 2 points)
            if len(x) >= 2 and len(y) >= 2:
                # slope, intercept, r_value, p_value, std_err
                slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
                r_squared = r_value**2

                regression_results.append({
                    'Replicate Group': base_rep,
                    'Time Slice (min)': f'{slice_start_time:.0f}-{slice_end_time:.0f}',
                    'Number of Points': points_per_slice,
                    'R-squared': r_squared,
                    'Slope': slope
                })

# Create a DataFrame from the results if any were generated
if regression_results:
    df_regression_summary = pd.DataFrame(regression_results)

    # Sort the DataFrame by 'Slope' in descending order
    df_regression_summary_sorted = df_regression_summary.sort_values(by='Slope', ascending=False).reset_index(drop=True)

    print("\n--- Linear Regression Analysis for All Possible Slices (Sorted by Slope) ---")

    # Get unique replicate groups to display separate tables
    unique_replicate_groups = df_regression_summary_sorted['Replicate Group'].unique()

    for group in unique_replicate_groups:
        group_df = df_regression_summary_sorted[df_regression_summary_sorted['Replicate Group'] == group]
        print(f"\n--- Results for Replicate Group: {group} ---")
        # Display the DataFrame with formatted R-squared and Slope values
        display(group_df.style.format({'R-squared': '{:.4f}', 'Slope': '{:.4f}'}))

else:
    print("No regression results to display. Please check your data and slice parameters.")

In [ ]:
#@title Low Std Dev R-squared across highest slopes (where data is increasing consistently)


# --- User-configurable parameter ---
std_dev_cutoff = 0.001 # Set the desired standard deviation cutoff here
# ----------------------------------

# Ensure df_regression_summary_sorted is available
if 'df_regression_summary_sorted' not in locals() or df_regression_summary_sorted.empty:
    print("Error: df_regression_summary_sorted is not available or is empty. Please run the previous cell (gRjZ8rQUOMeX) that generates it.")
else:
    # Get unique replicate groups
    unique_replicate_groups = df_regression_summary_sorted['Replicate Group'].unique()

    all_results_html = [] # To collect HTML for display
    # The all_df_result_sets list is no longer needed here as df_result_set will be assigned directly from df_regression_summary_sorted

    for group in unique_replicate_groups:
        print(f"\n--- Processing Replicate Group: {group} ---")

        # Filter data for the current replicate group
        df_group = df_regression_summary_sorted[df_regression_summary_sorted['Replicate Group'] == group].sort_values(by='R-squared', ascending=False).reset_index(drop=True)

        selected_rows_data = []
        current_r_squared_values_for_std = []

        # Iterate through the R-squared sorted DataFrame for the current group
        for index, row in df_group.iterrows():
            r_squared_val = row['R-squared']

            # Tentatively add the current R-squared value
            current_r_squared_values_for_std.append(r_squared_val)

            # Calculate standard deviation. Handle the case of a single element (std dev is NaN)
            if len(current_r_squared_values_for_std) == 1:
                current_std_dev = 0.0
            else:
                current_std_dev = pd.Series(current_r_squared_values_for_std).std()

            # Check if the standard deviation condition is met using the cutoff variable
            if current_std_dev < std_dev_cutoff:
                selected_rows_data.append(row.to_dict())
            else:
                # If adding this R-squared value makes the std dev >= cutoff,
                # then this row should not be included, and we stop.
                current_r_squared_values_for_std.pop() # Remove the last added R-squared
                break

        if selected_rows_data:
            df_result_set_group = pd.DataFrame(selected_rows_data)

            # Recalculate the standard deviation for the final set
            final_std_dev = df_result_set_group['R-squared'].std()
            # Handle case where only one item is selected (std dev is NaN for a single item)
            if len(df_result_set_group) == 1:
                final_std_dev = 0.0

            # Create HTML output for the current group
            group_html = []
            group_html.append(f'<div style="flex: 1; min-width: 300px; margin: 10px; border: 1px solid #ddd; padding: 10px;">')
            group_html.append(f'<h4>Results for Replicate Group: {group} (Std Dev R-squared &lt; {std_dev_cutoff})</h4>')
            group_html.append(df_result_set_group.style.format({'R-squared': '{:.4f}', 'Slope': '{:.4f}'}).to_html(index=False))
            group_html.append(f'<p>Standard deviation of R-squared for this set: {final_std_dev:.4f}</p>')
            group_html.append(f'<p>Number of items in this set: {len(df_result_set_group)}</p>')
            group_html.append(f'</div>')
            all_results_html.append(''.join(group_html))

        else:
            # This case should only be reached if df_group was empty, or no R-squared met the condition
            all_results_html.append(f'<div style="flex: 1; min-width: 300px; margin: 10px; border: 1px solid #ddd; padding: 10px;">')
            all_results_html.append(f'<h4>Results for Replicate Group: {group} (Std Dev R-squared &lt; {std_dev_cutoff})</h4>')
            all_results_html.append(f'<p>No R-squared values found for {group} that satisfy the condition.</p>')
            all_results_html.append(f'</div>')

    # Display the filtered HTML results
    if all_results_html:
        combined_html = f'<div style="display: flex; flex-wrap: wrap; justify-content: flex-start;">{''.join(all_results_html)}</div>'
        display(HTML(combined_html))
    else:
        print("No regression results to display for any group based on the std_dev_cutoff.")

    # Assign df_result_set to the full df_regression_summary_sorted
    # to ensure all replicate groups are available for subsequent cells.
    df_result_set = df_regression_summary_sorted.copy()

In [ ]:
#@title Extract filtered samples into separate DataFrames (Low Std Dev R-squared) and Show data from higher to lower slopes



# --- User-configurable parameter (same as in PkpSGz82lyaq) ---
std_dev_cutoff = 0.001 # Set the desired R-squared standard deviation cutoff here
# ----------------------------------

if 'df_regression_summary_sorted' in locals() and not df_regression_summary_sorted.empty:
    print(f"\n--- Extracting individual filtered sample DataFrames (Std Dev R-squared < {std_dev_cutoff}) ---")

    # Initialize an empty dictionary to store the individual filtered DataFrames
    filtered_individual_sample_dataframes = {}

    # Get unique replicate groups from the sorted DataFrame
    unique_replicate_groups = df_regression_summary_sorted['Replicate Group'].unique()

    for group in unique_replicate_groups:
        # Filter data for the current replicate group and sort by R-squared (descending)
        # This is crucial for the standard deviation cutoff logic
        df_group_raw = df_regression_summary_sorted[
            df_regression_summary_sorted['Replicate Group'] == group
        ].sort_values(by='R-squared', ascending=False).reset_index(drop=True)

        selected_rows_data = []
        current_r_squared_values_for_std = []

        # Iterate through the R-squared sorted DataFrame for the current group
        for index, row in df_group_raw.iterrows():
            r_squared_val = row['R-squared']

            # Tentatively add the current R-squared value
            current_r_squared_values_for_std.append(r_squared_val)

            # Calculate standard deviation. Handle the case of a single element (std dev is NaN)
            if len(current_r_squared_values_for_std) == 1:
                current_std_dev = 0.0
            else:
                current_std_dev = pd.Series(current_r_squared_values_for_std).std()

            # Check if the standard deviation condition is met
            if current_std_dev < std_dev_cutoff:
                selected_rows_data.append(row.to_dict())
            else:
                # If adding this R-squared value makes the std dev >= cutoff,
                # then this row should not be included, and we stop.
                current_r_squared_values_for_std.pop() # Remove the last added R-squared
                break

        if selected_rows_data:
            df_filtered_group = pd.DataFrame(selected_rows_data)
            # Drop the 'Replicate Group' column as it's now redundant within the individual DataFrame
            df_filtered_group = df_filtered_group.drop(columns=['Replicate Group']).reset_index(drop=True)

            # Store the filtered DataFrame in the dictionary
            filtered_individual_sample_dataframes[group] = df_filtered_group

            print(f"\nFiltered DataFrame for '{group}':")
            # display(df_filtered_group.style.format({'R-squared': '{:.4f}', 'Slope': '{:.4f}'}))

        else:
            print(f"\nNo filtered data found for '{group}' with R-squared standard deviation < {std_dev_cutoff}.")

    print("\nAll individual filtered sample DataFrames have been created and are accessible via the 'filtered_individual_sample_dataframes' dictionary.")
    print(f"Keys in 'filtered_individual_sample_dataframes': {list(filtered_individual_sample_dataframes.keys())}")

else:
    print("df_regression_summary_sorted is not available or is empty. Please run the previous cell that generates it.")

# import pandas as pd
# from IPython.display import HTML, display # Import HTML for rich display

if 'filtered_individual_sample_dataframes' in locals() and filtered_individual_sample_dataframes:
    print("\n--- Sorting each filtered sample DataFrame by Slope (descending) and displaying side-by-side ---")

    all_tables_html = [] # List to collect HTML representations of tables

    for group_name, df_sample in filtered_individual_sample_dataframes.items():
        if not df_sample.empty:
            # Sort the DataFrame by 'Slope' in descending order
            df_sorted_by_slope = df_sample.sort_values(by='Slope', ascending=False).reset_index(drop=True)

            # Convert each styled DataFrame to HTML
            table_html = df_sorted_by_slope.style.format({'R-squared': '{:.4f}', 'Slope': '{:.4f}'}).to_html(index=False)

            # Add a title above each table and wrap it in a div for layout
            all_tables_html.append(f'<div style="flex: 1; min-width: 300px; margin: 10px; border: 1px solid #ddd; padding: 10px;">')
            all_tables_html.append(f'<h4>Sorted Results for Replicate Group: {group_name}</h4>')
            all_tables_html.append(table_html)
            all_tables_html.append(f'</div>')
        else:
            # Add a placeholder for empty DataFrames to maintain layout if needed, or just print
            all_tables_html.append(f'<div style="flex: 1; min-width: 300px; margin: 10px; border: 1px solid #ddd; padding: 10px;">')
            all_tables_html.append(f'<h4>Sorted Results for Replicate Group: {group_name}</h4>')
            all_tables_html.append(f'<p>DataFrame is empty and cannot be sorted.</p>')
            all_tables_html.append(f'</div>')

    # Combine all individual table HTMLs into a single flex container
    if all_tables_html:
        combined_html = f'<div style="display: flex; flex-wrap: wrap; justify-content: flex-start;">{" ".join(all_tables_html)}</div>'
        display(HTML(combined_html))
    else:
        print("No filtered individual sample DataFrames were found to sort and display.")

else:
    print("filtered_individual_sample_dataframes is not available or is empty. Please run the previous cell that generates it.")


In [ ]:
#@title Highest slope for each sample in the filtered tables


# Ensure filtered_individual_sample_dataframes and df_summary_stats_multiindex are available
if 'filtered_individual_sample_dataframes' not in locals() or not filtered_individual_sample_dataframes:
    print("Error: filtered_individual_sample_dataframes is not available or is empty. Please run the preceding cells that generate it.")
elif 'df_summary_stats_multiindex' not in locals() or df_summary_stats_multiindex.empty:
    print("Error: df_summary_stats_multiindex is not available or is empty. Please run the previous data processing cells.")
else:
    print("--- Highest Slope for Each Replicate Group from Filtered DataFrames ---")

    highest_slopes_from_filtered = []

    for group_name, df_group in filtered_individual_sample_dataframes.items():
        if not df_group.empty:
            # Find the row with the maximum slope in the current filtered DataFrame
            # Explicitly sort by 'Slope' to guarantee finding the highest slope.
            highest_slope_row_for_group = df_group.sort_values(by='Slope', ascending=False).iloc[0].to_dict()
            highest_slope_row_for_group['Replicate Group'] = group_name # Add group name back for clarity
            highest_slopes_from_filtered.append(highest_slope_row_for_group)

    if highest_slopes_from_filtered:
        df_highest_slopes = pd.DataFrame(highest_slopes_from_filtered)
        # Reorder columns to have 'Replicate Group' first for better display
        cols = ['Replicate Group'] + [col for col in df_highest_slopes.columns if col != 'Replicate Group']
        df_highest_slopes = df_highest_slopes[cols]

        # Display the results with formatting
        display(df_highest_slopes.style.format({'R-squared': '{:.4f}', 'Slope': '{:.4f}'}))

        # Store the result in a variable for potential further use
        highest_slopes_per_group = df_highest_slopes

        # --- Plotting each highest slope in small subplots ---
        if not df_highest_slopes.empty:
            num_plots = len(df_highest_slopes)
            # Calculate grid dimensions for subplots (roughly square)
            num_cols = int(math.ceil(math.sqrt(num_plots)))
            num_rows = int(math.ceil(num_plots / num_cols))

            fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 5, num_rows * 4), squeeze=False)
            axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

            # Ensure 'Time [min]' in df_summary_stats_multiindex is numeric for comparison
            df_summary_stats_multiindex['Time [min]'] = pd.to_numeric(df_summary_stats_multiindex['Time [min]'], errors='coerce')

            for i, (index, row) in enumerate(df_highest_slopes.iterrows()):
                ax = axes[i]
                replicate_group = row['Replicate Group']
                slice_time_label = row['Time Slice (min)']

                # Extract start and end time from slice_time_label
                time_match = re.search(r'(\d+)-(\d+)', slice_time_label)
                if time_match:
                    start_time = int(time_match.group(1))
                    end_time = int(time_match.group(2))
                else:
                    print(f"Warning: Could not parse time slice label: {slice_time_label}. Using full range for {replicate_group}.")
                    start_time = df_summary_stats_multiindex['Time [min]'].min()
                    end_time = df_summary_stats_multiindex['Time [min]'].max()

                # Filter df_summary_stats_multiindex to get the data for the selected slice
                slice_data_for_plot = df_summary_stats_multiindex[
                    (df_summary_stats_multiindex['Time [min]'] >= start_time) &
                    (df_summary_stats_multiindex['Time [min]'] <= end_time)
                ].copy()

                x_data = slice_data_for_plot['Time [min]'].astype(float)
                y_data = slice_data_for_plot[(replicate_group, 'Avg (x100)')].astype(float)

                if len(x_data) < 2 or len(y_data) < 2:
                    ax.set_title(f'{replicate_group} ({slice_time_label})\nNo data to plot')
                    ax.set_visible(False) # Hide empty subplot
                else:
                    # Recalculate regression for the exact slice data for plotting accuracy
                    slope, intercept, r_value, p_value, std_err = stats.linregress(x_data, y_data)
                    r_squared = r_value**2

                    # Generate the best-fit line
                    x_fit = np.array([x_data.min(), x_data.max()])
                    y_fit = slope * x_fit + intercept

                    # Plot original data points
                    ax.scatter(x_data, y_data, color='blue', s=20, label='Data')
                    # Plot best-fit line
                    ax.plot(x_fit, y_fit, color='red', linestyle='--', label='Fit')

                    ax.set_title(f'{replicate_group} ({slice_time_label})\nSlope={slope:.2f}, R2={r_squared:.2f}', fontsize=9)
                    ax.set_xlabel('Time [min]', fontsize=8)
                    ax.set_ylabel('Avg (x100)', fontsize=8)
                    ax.grid(True)
                    ax.tick_params(axis='both', which='major', labelsize=7)

            # Hide any unused subplots
            for j in range(i + 1, len(axes)):
                axes[j].set_visible(False)

            plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to prevent title overlap
            fig.suptitle('Highest Slope Slices for Each Replicate Group (Filtered)', fontsize=16)
            plt.show()
        else:
            print("No highest slope results from filtered data to plot.")
    else:
        print("No highest slopes found from the filtered individual sample DataFrames to plot.")


In [ ]:
#@title IGNORE THIS
# #@title Choose Slice(s) to display

# import matplotlib.pyplot as plt
# from scipy import stats
# import pandas as pd
# import numpy as np # Make sure numpy is imported for np.array
# import re # Make sure re is imported for re.search

# # Ensure df_regression_summary_sorted and df_summary_stats_multiindex are available
# # In previous cells, df_result_set was assigned df_regression_summary_sorted.copy()
# # We use df_regression_summary_sorted directly here as it holds all results before std_dev filtering
# if 'df_regression_summary_sorted' not in locals() or df_regression_summary_sorted.empty:
#     print("Error: df_regression_summary_sorted is not available or is empty. Please run the previous cell (PkpSGz82lyaq) that generates it.")
# elif 'df_summary_stats_multiindex' not in locals() or df_summary_stats_multiindex.empty:
#     print("Error: df_summary_stats_multiindex is not available or is empty. Please run the previous data processing cells.")
# else:
#     # Get std_dev_cutoff from the previous cell, or use a default if not found
#     # This parameter is defined in cell PkpSGz82lyaq
#     std_dev_cutoff = locals().get('std_dev_cutoff', 0.001)
#     print(f"Using R-squared standard deviation cutoff: {std_dev_cutoff}")

#     # Get unique replicate groups from the full sorted regression summary
#     available_replicate_groups = df_regression_summary_sorted['Replicate Group'].unique().tolist()
#     print("Available Replicate Groups:", available_replicate_groups)

#     # Step 1: User chooses a Replicate Group
#     selected_group = None
#     while selected_group not in available_replicate_groups:
#         selected_group = input("Enter the Replicate Group you want to visualize: ")
#         if selected_group not in available_replicate_groups:
#             print(f"Invalid Replicate Group. Please choose from: {', '.join(available_replicate_groups)}")

#     # Filter df_regression_summary_sorted for the selected group
#     df_group_raw = df_regression_summary_sorted[
#         df_regression_summary_sorted['Replicate Group'] == selected_group
#     ].sort_values(by='R-squared', ascending=False).reset_index(drop=True)

#     # --- Apply the std_dev_cutoff filtering logic for the selected group, mirroring PkpSGz82lyaq ---
#     selected_rows_data_for_plot = []
#     current_r_squared_values_for_std = []

#     for index, row in df_group_raw.iterrows():
#         r_squared_val = row['R-squared']

#         current_r_squared_values_for_std.append(r_squared_val)

#         # Calculate standard deviation. Handle the case of a single element (std dev is NaN)
#         if len(current_r_squared_values_for_std) == 1:
#             current_std_dev = 0.0
#         else:
#             current_std_dev = pd.Series(current_r_squared_values_for_std).std()

#         # Check if the standard deviation condition is met
#         if current_std_dev < std_dev_cutoff:
#             selected_rows_data_for_plot.append(row.to_dict())
#         else:
#             # If adding this R-squared value makes the std dev >= cutoff, then this row should not be included, and we stop.
#             current_r_squared_values_for_std.pop() # Remove the last added R-squared
#             break

#     # This df_filtered_by_group now contains the slices that passed the std_dev_cutoff for the selected group
#     df_filtered_by_group = pd.DataFrame(selected_rows_data_for_plot)

#     if df_filtered_by_group.empty:
#         print(f"No results found for the selected Replicate Group: {selected_group} with R-squared standard deviation less than {std_dev_cutoff}.")
#     else:
#         print(f"\n--- Available slices for {selected_group} (filtered by Std Dev R-squared < {std_dev_cutoff}) ---")
#         # The user requested to remove the display of the table
#         # display(df_filtered_by_group.style.format({'R-squared': '{:.4f}', 'Slope': '{:.4f}'}))

#         # Step 2: User chooses row numbers from the filtered table
#         while True:
#             try:
#                 row_num_input_str = input(f"\nEnter row numbers (indices) from the above summary table for {selected_group}, comma-separated (e.g., '0,1') (0 to {len(df_filtered_by_group) - 1}): ")
#                 row_indices_str = [x.strip() for x in row_num_input_str.split(',')]
#                 row_indices = [int(x) for x in row_indices_str if x] # Convert to int, handle empty strings after split

#                 valid_indices = True
#                 if not row_indices:
#                     print("No row numbers entered. Please try again.")
#                     valid_indices = False
#                 else:
#                     for idx in row_indices:
#                         if not (0 <= idx < len(df_filtered_by_group)):
#                             print(f"Invalid row number: {idx}. Please enter indices within the displayed range for {selected_group}.")
#                             valid_indices = False
#                             break
#                 if valid_indices:
#                     break
#             except ValueError:
#                 print("Invalid input. Please enter comma-separated integers.")

#         plt.figure(figsize=(12, 7))
#         plt.title(f'Multiple Regression Slices and Fits for {selected_group}')
#         plt.xlabel('Time [min]')
#         plt.ylabel('Average of Sample (x100)') # Keep y-axis label consistent
#         plt.grid(True)

#         # Use a colormap to get distinct colors for multiple plots
#         colors = plt.cm.get_cmap('tab10', len(row_indices))

#         for i, row_index in enumerate(row_indices):
#             current_color = colors(i) # Get a unique color for each slice

#             # Get the selected regression result row from the filtered DataFrame
#             selected_row = df_filtered_by_group.iloc[row_index]

#             slice_time_label = selected_row['Time Slice (min)']
#             # The replicate_group is already known (selected_group)
#             r_squared_from_summary = selected_row['R-squared']
#             slope_from_summary = selected_row['Slope']

#             print(f"\nPreparing plot for: Row ID {row_index}, Time Slice: {slice_time_label}")

#             # Extract start and end time from slice_time_label
#             try:
#                 start_time_str, end_time_str = slice_time_label.split('-')
#                 start_time = int(start_time_str)
#                 end_time = int(end_time_str)
#             except ValueError:
#                 print(f"Error parsing time slice label: {slice_time_label}. Attempting to use full range.")
#                 start_time = df_summary_stats_multiindex['Time [min]'].min()
#                 end_time = df_summary_stats_multiindex['Time [min]'].max()

#             # Filter df_summary_stats_multiindex to get the data for the selected slice
#             # Ensure 'Time [min]' in df_summary_stats_multiindex is numeric for comparison
#             df_summary_stats_multiindex['Time [min]'] = pd.to_numeric(df_summary_stats_multiindex['Time [min]'], errors='coerce')

#             slice_data_for_plot = df_summary_stats_multiindex[
#                 (df_summary_stats_multiindex['Time [min]'] >= start_time) &
#                 (df_summary_stats_multiindex['Time [min]'] <= end_time)
#             ].copy()

#             x_data = slice_data_for_plot['Time [min]'].astype(float)
#             y_data = slice_data_for_plot[(selected_group, 'Avg (x100)')].astype(float) # Use the selected_group for column name

#             if len(x_data) < 2 or len(y_data) < 2:
#                 print(f"Error: Not enough data points to plot or perform regression for the selected slice (Row ID: {row_index}).")
#             else:
#                 # Recalculate regression for the exact slice data (to ensure fit line is accurate)
#                 slope, intercept, r_value, p_value, std_err = stats.linregress(x_data, y_data)
#                 r_squared = r_value**2

#                 # Generate the best-fit line
#                 x_fit = np.array([x_data.min(), x_data.max()])
#                 y_fit = slope * x_fit + intercept

#                 # Plot original data points
#                 plt.scatter(x_data, y_data, label=f'Row {row_index}: Data ({slice_time_label})', color=current_color, s=50, zorder=5)

#                 # Plot best-fit line
#                 plt.plot(x_fit, y_fit, color=current_color, linestyle='--', label=f'Row {row_index}: Fit (Slope={slope:.2f}, R2={r_squared:.2f})')

#         plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') # Place legend outside the plot
#         plt.tight_layout() # Adjust layout to prevent legend overlap
#         plt.show()

In [ ]:
#@title Shows where each plot is on the overall graph

# Ensure df_summary_stats_multiindex and highest_slopes_per_group are available
if 'df_summary_stats_multiindex' not in locals() or df_summary_stats_multiindex.empty:
    print("Error: df_summary_stats_multiindex is not available or is empty. Please run the data processing cells.")
elif 'highest_slopes_per_group' not in locals() or highest_slopes_per_group.empty:
    print("Error: highest_slopes_per_group is not available or is empty. Please run the preceding cells that generate it.")
else:
    plt.figure(figsize=(15, 9))
    plt.title('Overall Data with Highlighted Highest Slope Slices', fontsize=16)
    plt.xlabel('Time [min]', fontsize=12)
    plt.ylabel('Average of Sample (x100)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)

    # Get a list of unique replicate group names from the MultiIndex for consistent coloring
    replicate_groups_for_coloring = [col[0] for col in df_summary_stats_multiindex.columns if col[1] == 'Avg (x100)']
    num_unique_replicate_groups = len(replicate_groups_for_coloring)

    # Retrieve the 'tab10' colormap object directly
    cmap_object = plt.colormaps['tab10']

    # Create a dictionary to map each replicate group name to a specific color
    # This ensures consistency between overall plot and highlight plot, cycling colors if needed
    group_color_dict = {group_name: cmap_object(i % cmap_object.N)
                        for i, group_name in enumerate(replicate_groups_for_coloring)}

    # Plot all data points for each replicate group first
    for base_rep in replicate_groups_for_coloring: # Iterate through unique base reps
        current_group_color = group_color_dict[base_rep]
        plt.plot(
            df_summary_stats_multiindex['Time [min]'],
            df_summary_stats_multiindex[(base_rep, 'Avg (x100)')],
            label=f'{base_rep} Overall Average',
            color=current_group_color,
            alpha=0.3, # Make overall lines more subtle
            linewidth=1,
            zorder=1 # Ensure overall lines are in background
        )

    # Now, iterate through highest_slopes_per_group to highlight the best slices
    for idx, (index, row) in enumerate(highest_slopes_per_group.iterrows()):
        replicate_group = row['Replicate Group']
        slice_time_label = row['Time Slice (min)']
        slope = row['Slope']
        r_squared = row['R-squared']

        # Extract start and end time from slice_time_label
        time_match = re.search(r'(\d+)-(\d+)', slice_time_label)
        if time_match:
            start_time = int(time_match.group(1))
            end_time = int(time_match.group(2))
        else:
            print(f"Warning: Could not parse time slice label: {slice_time_label} for {replicate_group}. Skipping highlighting.")
            continue

        # Filter the original df_summary_stats_multiindex to get the data for this specific slice
        slice_data_for_highlight = df_summary_stats_multiindex[
            (df_summary_stats_multiindex['Time [min]'] >= start_time) &
            (df_summary_stats_multiindex['Time [min]'] <= end_time)
        ].copy()

        x_highlight = slice_data_for_highlight['Time [min]'].astype(float)
        y_highlight = slice_data_for_highlight[(replicate_group, 'Avg (x100)')].astype(float)

        if len(x_highlight) >= 2 and len(y_highlight) >= 2:
            # Use the consistent color from the dictionary
            highlight_color = group_color_dict.get(replicate_group, 'black') # Fallback color if group not found

            # Plot the highlighted segment
            plt.plot(
                x_highlight,
                y_highlight,
                color=highlight_color,
                linewidth=5, # Make highlight thicker
                linestyle='-',
                label=f'{replicate_group} Highest Slope ({slice_time_label})\nSlope={slope:.2f}, R2={r_squared:.2f}',
                zorder=2 # Ensure highlighted line is on top
            )
            # Add a vertical line at the start and end of the slice for clarity
            plt.axvline(x=start_time, color=highlight_color, linestyle=':', linewidth=1, alpha=0.7, zorder=0)
            plt.axvline(x=end_time, color=highlight_color, linestyle=':', linewidth=1, alpha=0.7, zorder=0)

    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
#@title Choose a sample to better visualize the choosen time slice


# Ensure df_summary_stats_multiindex and highest_slopes_per_group are available
if 'df_summary_stats_multiindex' not in locals() or df_summary_stats_multiindex.empty:
    print("Error: df_summary_stats_multiindex is not available or is empty. Please run the data processing cells.")
elif 'highest_slopes_per_group' not in locals() or highest_slopes_per_group.empty:
    print("Error: highest_slopes_per_group is not available or is empty. Please run the preceding cells that generate it.")
else:
    # Get a list of unique replicate group names from the MultiIndex for user selection
    available_replicate_groups = [col[0] for col in df_summary_stats_multiindex.columns if isinstance(col, tuple) and col[1] == 'Avg (x100)']

    if not available_replicate_groups:
        print("No replicate groups found for plotting.")
    else:
        print("Available Replicate Groups:")
        for i, group in enumerate(available_replicate_groups):
            print(f"{i+1}. {group}")

        selected_group = None
        while True:
            user_input = input("Enter the number corresponding to the replicate group you want to plot: ")
            try:
                selection_index = int(user_input) - 1
                if 0 <= selection_index < len(available_replicate_groups):
                    selected_group = available_replicate_groups[selection_index]
                    break
                else:
                    print("Invalid number. Please enter a number from the list.")
            except ValueError:
                print("Invalid input. Please enter a number.")

        plt.figure(figsize=(12, 7))

        # Plot the overall average for the selected group
        avg_col_name = (selected_group, 'Avg (x100)')
        plt.plot(df_summary_stats_multiindex['Time [min]'], df_summary_stats_multiindex[avg_col_name], label=f'{selected_group} Overall Average', color='blue', alpha=0.6)

        # Find the highest slope data for the selected group
        highest_slope_row = highest_slopes_per_group[
            highest_slopes_per_group['Replicate Group'] == selected_group
        ]

        if not highest_slope_row.empty:
            # Assuming there's only one highest slope per group in highest_slopes_per_group
            highest_slope_data = highest_slope_row.iloc[0]
            slice_time_label = highest_slope_data['Time Slice (min)']
            slope = highest_slope_data['Slope']
            r_squared = highest_slope_data['R-squared']

            # Extract start and end time from slice_time_label
            time_match = re.search(r'(\d+)-(\d+)', slice_time_label)
            if time_match:
                start_time = int(time_match.group(1))
                end_time = int(time_match.group(2))
            else:
                print(f"Warning: Could not parse time slice label: {slice_time_label}. Cannot highlight slice.")
                start_time, end_time = None, None

            if start_time is not None and end_time is not None:
                # Filter df_summary_stats_multiindex to get the data for the selected slice
                slice_data_for_highlight = df_summary_stats_multiindex[
                    (df_summary_stats_multiindex['Time [min]'] >= start_time) &
                    (df_summary_stats_multiindex['Time [min]'] <= end_time)
                ].copy()

                x_highlight = slice_data_for_highlight['Time [min]'].astype(float)
                y_highlight = slice_data_for_highlight[avg_col_name].astype(float)

                if len(x_highlight) >= 2:
                    plt.plot(
                        x_highlight,
                        y_highlight,
                        color='red', # Highlight color
                        linewidth=3, # Thicker line for highlight
                        linestyle='-',
                        label=f'Highest Slope Slice ({slice_time_label})\nSlope={slope:.2f}, R2={r_squared:.2f}',
                        zorder=2 # Ensure highlight is on top
                    )
                    # Add vertical lines to delineate the slice
                    plt.axvline(x=start_time, color='gray', linestyle=':', linewidth=1, alpha=0.7)
                    plt.axvline(x=end_time, color='gray', linestyle=':', linewidth=1, alpha=0.7)

        plt.xlabel('Time [min]')
        plt.ylabel('Average of Sample (x100)')
        plt.title(f'Overall Graph for {selected_group} with Highest Slope Slice Highlighted')
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
#@title Normalized Slope Bar Plot for Top Slice per Replicate Group


# Ensure highest_slopes_per_group is available from previous steps
if 'highest_slopes_per_group' not in locals() or highest_slopes_per_group.empty:
    print("Error: highest_slopes_per_group is not available or is empty. Please run the preceding cells that generate it.")
else:
    # Use highest_slopes_per_group directly as it already contains the highest slopes for each group from the filtered data
    df_top_slopes_per_group = highest_slopes_per_group.copy()

    # Find the N2 15 reference slope for normalization
    print("Available Replicate Groups:", available_replicate_groups)
    replicate_input = input("Enter your replicate group here:")
    n2_15_reference_data = df_top_slopes_per_group[
        df_top_slopes_per_group['Replicate Group'] == replicate_input # Modify HERE 🔧
    ]

    if n2_15_reference_data.empty:
        print("Error: 'N2 DMSO' reference group not found in the highest slopes summary. Cannot perform normalization.")
    else:
        # Ensure we take the *single* highest slope for N2 DMSO, in case there were multiple N2 DMSO groups somehow
        n2_15_reference_data = n2_15_reference_data.sort_values(by='Slope', ascending=False).iloc[0]

        n2_15_ref_slope = n2_15_reference_data['Slope']
        n2_15_ref_slice_label = n2_15_reference_data['Time Slice (min)'] # Fixed column name
        n2_15_ref_group = n2_15_reference_data['Replicate Group']

        if n2_15_ref_slope == 0:
            print("Error: The reference 'N2 DMSO' slope is zero. Cannot perform normalization. Please check your data.")
        else:
            # Prepare a list to store data for the bar plot
            plot_data = []

            # Add the dynamically selected top samples, normalized by the 'N2 DMSO' reference slope
            for index, row in df_top_slopes_per_group.iterrows():
                current_group = row['Replicate Group']
                current_slice = row['Time Slice (min)'] # Fixed column name
                current_slope = row['Slope']

                normalized_slope = current_slope / n2_15_ref_slope

                plot_data.append({
                    'Sample Label': f"{current_group} ({current_slice})",
                    'Original Slope': current_slope,
                    'Normalized Slope': normalized_slope
                })

            # Create a DataFrame from the prepared plot data
            df_bar_plot = pd.DataFrame(plot_data)

            # Create the bar plot
            plt.figure(figsize=(12, 7))

            # Define colors for the bars - all will be 'skyblue'
            colors = ['skyblue'] * len(df_bar_plot)

            bars = plt.bar(df_bar_plot['Sample Label'], df_bar_plot['Normalized Slope'], color=colors)

            plt.xlabel('Sample and Slice')
            plt.ylabel(f'Normalized Slope (Relative to {n2_15_ref_group} {n2_15_ref_slice_label})')
            plt.title('Normalized Slopes for Top Slices of Each Sample Group (from Filtered Data)')
            plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout() # Adjust layout to prevent labels from overlapping
            plt.show()

            print("\nOriginal and Normalized Slope Values:")
            display(df_bar_plot.style.format({'Original Slope': '{:.4f}', 'Normalized Slope': '{:.4f}'}))

# Export Data and Graphs into Excel

1. Run the cell below

2. Enter the name for your file

3. Open the folder in Colab by clicking the folder icon 📁 on the right side of the screen

4. Hit the refresh button 🔃

5. Hover over the file that just got exported, then click the 3 dot, then download

In [ ]:
#@title Run this cell to export data and graphs

# --- 1. Regenerate and save the 'Overall Data with Highlighted Highest Slope Slices' graph ---
# Ensure df_summary_stats_multiindex and highest_slopes_per_group are available from previous cells.

plt.figure(figsize=(15, 9))
plt.title('Overall Data with Highlighted Highest Slope Slices', fontsize=16)
plt.xlabel('Time [min]', fontsize=12)
plt.ylabel('Average of Sample (x100)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# Get a list of unique replicate group names from the MultiIndex for consistent coloring
replicate_groups_for_coloring = [col[0] for col in df_summary_stats_multiindex.columns if col[1] == 'Avg (x100)']

# Retrieve the 'tab10' colormap object directly
cmap_object = plt.colormaps['tab10']

# Create a dictionary to map each replicate group name to a specific color
group_color_dict = {group_name: cmap_object(i % cmap_object.N)
                    for i, group_name in enumerate(replicate_groups_for_coloring)}

# Plot all data points for each replicate group first
for base_rep in replicate_groups_for_coloring:
    current_group_color = group_color_dict[base_rep]
    plt.plot(
        df_summary_stats_multiindex['Time [min]'],
        df_summary_stats_multiindex[(base_rep, 'Avg (x100)')],
        label=f'{base_rep} Overall Average',
        color=current_group_color,
        alpha=0.3,
        linewidth=1,
        zorder=1
    )

# Iterate through highest_slopes_per_group to highlight the best slices
for idx, (index, row) in enumerate(highest_slopes_per_group.iterrows()):
    replicate_group = row['Replicate Group']
    slice_time_label = row['Time Slice (min)']
    slope = row['Slope']
    r_squared = row['R-squared']

    # Extract start and end time from slice_time_label
    time_match = re.search(r'(\d+)-(\d+)', slice_time_label)
    if time_match:
        start_time = int(time_match.group(1))
        end_time = int(time_match.group(2))
    else:
        print(f"Warning: Could not parse time slice label: {slice_time_label} for {replicate_group}. Skipping highlighting.")
        continue

    # Filter the original df_summary_stats_multiindex to get the data for this specific slice
    slice_data_for_highlight = df_summary_stats_multiindex[
        (df_summary_stats_multiindex['Time [min]'] >= start_time) &
        (df_summary_stats_multiindex['Time [min]'] <= end_time)
    ].copy()

    x_highlight = slice_data_for_highlight['Time [min]'].astype(float)
    y_highlight = slice_data_for_highlight[(replicate_group, 'Avg (x100)')].astype(float)

    if len(x_highlight) >= 2 and len(y_highlight) >= 2:
        highlight_color = group_color_dict.get(replicate_group, 'black')

        plt.plot(
            x_highlight,
            y_highlight,
            color=highlight_color,
            linewidth=5,
            linestyle='-',
            label=f'{replicate_group} Highest Slope ({slice_time_label})\nSlope={slope:.2f}, R2={r_squared:.2f}',
            zorder=2
        )
        plt.axvline(x=start_time, color=highlight_color, linestyle=':', linewidth=1, alpha=0.7, zorder=0)
        plt.axvline(x=end_time, color=highlight_color, linestyle=':', linewidth=1, alpha=0.7, zorder=0)

plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=10)
plt.tight_layout()
overall_graph_filename = 'overall_graph_highest_slopes.png'
plt.savefig(overall_graph_filename, bbox_inches='tight')
plt.close()

# --- 2. Regenerate and save the 'Normalized Slope Bar Plot' ---
# This assumes df_bar_plot, n2_15_ref_group, and n2_15_ref_slice_label are available from previous cells.

plt.figure(figsize=(12, 7))

colors = ['skyblue'] * len(df_bar_plot)

plt.bar(df_bar_plot['Sample Label'], df_bar_plot['Normalized Slope'], color=colors)

plt.xlabel('Sample and Slice')
plt.ylabel(f'Normalized Slope (Relative to {n2_15_ref_group} {n2_15_ref_slice_label})')
plt.title('Normalized Slopes for Top Slices of Each Sample Group (from Filtered Data)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
bar_plot_filename = 'normalized_slope_bar_plot.png'
plt.savefig(bar_plot_filename, bbox_inches='tight')
plt.close()

# --- 3. Create Excel file and embed data/graphs on a single sheet ---
name = input("What would you like to name your file?")
excel_filename = f'{name}.xlsx'
with pd.ExcelWriter(excel_filename, engine='xlsxwriter') as writer:
    # Get the xlsxwriter workbook and worksheet objects.
    workbook = writer.book
    summary_worksheet = workbook.add_worksheet('Summary')

    # Write the df_bar_plot table to the 'Summary' sheet
    # Start at row 0, column 0 (A1)
    df_bar_plot.to_excel(writer, sheet_name='Summary', startrow=0, startcol=0, index=False)

    # Calculate the column for the bar plot image (next to the table)
    bar_plot_col = df_bar_plot.shape[1] + 1 # +1 for a buffer column

    # Add the Normalized Slope Bar Plot image to the 'Summary' sheet with scaling
    summary_worksheet.insert_image(0, bar_plot_col, bar_plot_filename, {'xscale': 0.1, 'yscale': 0.1})

    # Calculate the starting row for the second image, leaving space after the table
    # +1 for header row, +3 for buffer rows
    overall_graph_start_row = df_bar_plot.shape[0] + 1 + 3

    # Add the Overall Data with Highlighted Highest Slope Slices graph to the 'Summary' sheet with scaling
    summary_worksheet.insert_image(overall_graph_start_row, 0, overall_graph_filename, {'xscale': 0.1, 'yscale': 0.1})

print(f"Analysis results exported to '{excel_filename}'")